# EDGE — 90s Talking-Head Explainer (Kaggle / SadTalker)

Generates a real lip-synced talking-head video from ONE profile photo + script.

**Model:** SadTalker (image + audio → talking head), run on Kaggle free GPU.

**Outputs:** `edge_company_intro_90sec.mp4`, `.srt`, `_script.txt`, `README.md`

**Before running:** Settings → Accelerator = GPU (T4/P100). Internet must be ON (default).

In [ ]:
# 0. Install SadTalker + deps (Kaggle Python 3.12 + CUDA)
import os, subprocess, sys
from IPython.display import clear_output

if not os.path.exists('/kaggle/working/SadTalker'):
    !git clone https://github.com/OpenTalker/SadTalker.git /kaggle/working/SadTalker
os.chdir('/kaggle/working/SadTalker')

for pkg in [
    'numpy==1.26.4','scipy==1.13.1','imageio==2.34.0',
    'imageio-ffmpeg==0.5.1','pydub==0.25.1','resampy==0.3.1','joblib==1.4.2',
    'librosa==0.10.2','numba==0.60.0','yacs==0.1.8','pyyaml','tqdm','av',
    'safetensors','kornia==0.7.2','dlib','face_alignment==1.3.5','basicsr==1.4.2',
    'facexlib==0.3.0','gfpgan','einops','opencv-python-headless','tensorboard'
]:
    try:
        subprocess.run([sys.executable,'-m','pip','install','-q',pkg],check=False,
                        stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    except Exception as e:
        print('skip',pkg,e)
# scikit-image 0.22 wheel is built against numpy 2.x (ABI 96) but we pin numpy 1.26 (ABI 88).
# Build it from source so its C extensions match numpy 1.26 (keeps basicsr/face_alignment happy).
subprocess.run([sys.executable,'-m','pip','install','-q','--no-binary','scikit-image','--force-reinstall','scikit-image==0.22.0'],check=False,
                stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
subprocess.run(['apt-get','-qq','install','-y','ffmpeg'],check=False,
                stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
print('deps done')
import numpy, skimage
print('numpy', numpy.__version__, 'skimage', skimage.__version__)


In [ ]:
# 1. Download SadTalker model checkpoints (one-time, ~few hundred MB)
os.chdir('/kaggle/working/SadTalker')
!bash scripts/download_models.sh
print('models downloaded' if os.path.exists('checkpoints/SadTalker_V0.0.2_512.safetensors') else 'MODEL DOWNLOAD FAILED')

In [ ]:
# 2. Fetch profile image (exact face reference) from Google Drive
import urllib.request, os
IMG_ID='1-2sFUEHqXDbaPq0lfBmamrjQBsdL_QuY'
img_url=f'https://drive.google.com/uc?export=download&id={IMG_ID}'
os.makedirs('/kaggle/working/assets',exist_ok=True)
urllib.request.urlretrieve(img_url,'/kaggle/working/assets/profile.jpg')
print('profile.jpg', os.path.getsize('/kaggle/working/assets/profile.jpg'),'bytes')

In [ ]:
# 3. Script (verbatim) + generate male voiceover (free TTS, 3x30s segments)
import urllib.request, urllib.parse, subprocess, os, textwrap

SCRIPT = '''
I'm Mirsina Aghdam, CEO of EDGE - Earthwise Dynamics Geo Environs. We're an Irish research-driven startup operating at the intersection of geoengineering, AI and space science.

Europe imports almost all the rare earth elements it needs for electric vehicles, wind turbines and defence systems. Current exploration methods are slow, expensive and miss low-grade deposits that could be strategic assets.

EDGE has developed TerraLens - an AI-enabled platform that combines drone-mounted gamma spectroscopy, LiDAR, satellite data and machine learning to locate and quantify rare earth mineralisation. We're funded by ESA Business Incubation Centre Ireland to build this technology.

Phase one is terrestrial - we're validating TerraLens through field surveys across European sites, proving the technology works on real geology. Phase two builds on that foundation to create a space-based version for lunar and asteroid prospecting. Russia and China are already moving on lunar resources. Europe needs this capability.

We're a team of senior engineers dedicated to pushing boundaries in geospatial modelling and automated AI solutions. If you're working on critical minerals, space resources or AI-driven exploration, let's talk.
'''

with open('/kaggle/working/assets/script.txt','w') as f:
    f.write(SCRIPT)

# Free TTS: split into chunks, fetch from Google translate endpoint, concat
words = SCRIPT.replace('\n',' ').split()
chunks=[]; cur=''
for w in words:
    if len(cur)+len(w) > 150: chunks.append(cur); cur=''
    cur += (' ' if cur else '') + w
if cur: chunks.append(cur)

for i,p in enumerate(chunks):
    u='https://translate.google.com/translate_tts?ie=UTF-8&client=tw-ob&tl=en-GB&q='+urllib.parse.quote(p)
    urllib.request.urlretrieve(u,f'/kaggle/working/assets/tts_{i}.mp3')
with open('/kaggle/working/assets/tts_list.txt','w') as f:
    f.write('\n'.join(f"file '/kaggle/working/assets/tts_{i}.mp3'" for i in range(len(chunks))))

subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i','/kaggle/working/assets/tts_list.txt',
                '-ar','16000','-ac','1','/kaggle/working/assets/voice.wav'],check=True)

# split into 3x30s
for i in range(3):
    start=i*30
    dur=31 if i==2 else 30
    subprocess.run(['ffmpeg','-y','-i','/kaggle/working/assets/voice.wav',
                    '-ss',str(start),'-t',str(dur),'-ar','16000','-ac','1',
                    f'/kaggle/working/assets/seg_{i}.wav'],check=True)
print('voice + segments ready')

In [ ]:
# 4. Generate talking head with SadTalker — one segment at a time (safer on free GPU)
import subprocess, os
os.chdir('/kaggle/working/SadTalker')
for i in range(3):
    cmd = ['python','inference.py',
           '--driven_audio', f'/kaggle/working/assets/seg_{i}.wav',
           '--source_image', '/kaggle/working/assets/profile.jpg',
           '--result_dir', '/kaggle/working/results',
           '--size','512','--preprocess','crop','--enhancer','gfpgan','--batch_size','1']
    print(f'--- segment {i} ---')
    subprocess.run(cmd, check=False)
print('sadtalker runs complete')

In [ ]:
# 5. Stitch the 3 segments + upscale to 1080p
import glob, os, subprocess
os.makedirs('/kaggle/working/final',exist_ok=True)
segs = sorted(glob.glob('/kaggle/working/results/*/*.mp4'))
print('found segments:', segs)
with open('/kaggle/working/final/concat.txt','w') as f:
    for s in segs: f.write(f"file '{s}'\n")
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i','/kaggle/working/final/concat.txt',
                '-c','copy','/kaggle/working/final/stitched.mp4'],check=True)
# upscale 512->1080 and normalize audio to 48k
subprocess.run(['ffmpeg','-y','-i','/kaggle/working/final/stitched.mp4',
                '-vf','scale=1920:1080:flags=lanczos','-r','30',
                '-c:v','libx264','-crf','20','-pix_fmt','yuv420p',
                '-ar','48000','-c:a','aac','-b:a','160k',
                '/kaggle/working/final/edge_company_intro_90sec.mp4'],check=True)
print('final video ready')

In [ ]:
# 6. Burn-in subtitles + 4 title cards via ffmpeg drawtext
import subprocess, os
OUT='/kaggle/working/final/edge_company_intro_90sec.mp4'
TMP='/kaggle/working/final/subtitled.mp4'
# Subtitle SRT
srt='''1
00:00:00,000 --> 00:00:14,000
I'm Mirsina Aghdam, CEO of EDGE - Earthwise Dynamics Geo Environs.

2
00:00:14,000 --> 00:00:30,000
Europe imports almost all the rare earth elements it needs.

3
00:00:30,000 --> 00:00:50,000
EDGE developed TerraLens: drone gamma spectroscopy, LiDAR, satellite, AI.

4
00:00:50,000 --> 00:01:15,000
Phase 1 terrestrial validation. Phase 2 space-based lunar and asteroid prospecting.

5
00:01:15,000 --> 00:01:30,000
Senior engineers building better decisions on Europe's mineral resources. Let's talk.
'''
open('/kaggle/working/final/subs.srt','w').write(srt)
cards = [
  "drawtext=text='Mirsina Aghdam | CEO, EDGE':fontcolor=white:fontsize=42:x=(w-tw)/2:y=h-120:enable='between(t,1,12)'",
  "drawtext=text='TerraLens AI | Rare Earth Exploration Intelligence':fontcolor=white:fontsize=38:x=(w-tw)/2:y=h-120:enable='between(t,30,48)'",
  "drawtext=text='Phase 1 | Terrestrial validation':fontcolor=white:fontsize=38:x=(w-tw)/2:y=h-120:enable='between(t,50,66)'",
  "drawtext=text='Phase 2 | Space-enabled mineral intelligence':fontcolor=white:fontsize=36:x=(w-tw)/2:y=h-120:enable='between(t,66,82)'",
]
filter_parts = [
  "subtitles='/kaggle/working/final/subs.srt':force_style='FontSize=22,PrimaryColour=&H00FFFFFF,BackColour=&H80000000'"
] + cards
vf = ','.join(filter_parts)
subprocess.run(['ffmpeg','-y','-i',OUT,'-vf',vf,'-c:v','libx264','-crf','20','-c:a','copy',TMP],check=True)
os.replace(TMP,OUT)
print('subtitles + title cards burned in')

In [ ]:
# 7. Copy deliverables to working root + write README
import shutil, os
src='/kaggle/working/final/edge_company_intro_90sec.mp4'
shutil.copy(src,'/kaggle/working/edge_company_intro_90sec.mp4')
shutil.copy('/kaggle/working/assets/script.txt','/kaggle/working/edge_company_intro_90sec_script.txt')
shutil.copy('/kaggle/working/final/subs.srt','/kaggle/working/edge_company_intro_90sec.srt')
readme='''EDGE 90s Talking-Head Explainer
Model: SadTalker (OpenTalker) on Kaggle free GPU
Resolution: 512 -> upscaled 1920x1080, 30fps, H.264, AAC 48kHz
Voice: free TTS (en-GB). Replace assets/voice.wav for custom accent.
Segments: 3x30s generated separately, stitched, upscaled, subtitled.
'''
open('/kaggle/working/README.md','w').write(readme)
print('deliverables in /kaggle/working/')
print([f for f in os.listdir('/kaggle/working') if f.startswith('edge_') or f=='README.md'])